In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score
import joblib

In [3]:
df = pd.read_csv('drom_archive_cleaned_2018-2025.csv')

In [4]:
df["Дата размещения объявления"] = pd.to_datetime(df["Дата размещения объявления"])
df["Год объявления"] = df["Дата размещения объявления"].dt.year
df["Месяц объявления"] = df["Дата размещения объявления"].dt.month
df["Возраст авто"] = df["Год объявления"] - df["Год"]
df = df.drop("Дата размещения объявления", axis=1)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 18 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   Название машины   1000000 non-null  str    
 1   Год               1000000 non-null  float64
 2   Цена              1000000 non-null  float64
 3   Объем двигателя   1000000 non-null  float64
 4   Тип двигателя     1000000 non-null  str    
 5   Мощность          1000000 non-null  float64
 6   Коробка передач   1000000 non-null  str    
 7   Привод            1000000 non-null  str    
 8   Пробег            1000000 non-null  float64
 9   Руль              1000000 non-null  str    
 10  Поколение         1000000 non-null  float64
 11  Рестайлинг        1000000 non-null  float64
 12  Тип кузова        1000000 non-null  str    
 13  Метка             1000000 non-null  str    
 14  Город             1000000 non-null  str    
 15  Год объявления    1000000 non-null  int32  
 16  Месяц объявл

In [5]:
categorical = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Поколение', 'Рестайлинг',
               'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Месяц объявления', 'Возраст авто']

In [6]:
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical), ('num', 'passthrough', numerical)], remainder='drop')

In [7]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=50,
        random_state=42,
        n_jobs=-1,
        max_depth=20,
        verbose=2
    ))
])

In [8]:
y = df['Цена']
X = df.drop('Цена', axis=1)

In [9]:
X_train, X_test, y_train, y_test = joblib.load("data_split.pkl")

In [ ]:
model.fit(X_train, y_train)
joblib.dump(model, "rf_model.pkl")

building tree 17 of 50
building tree 18 of 50


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
rf_mse = mean_squared_error(y_test, y_pred)
rf_rmse = np.sqrt(rf_mse)
rf_mae = mean_absolute_error(y_test, y_pred)
rf_r2 = r2_score(y_test, y_pred)

In [ ]:
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [rf_mse, rf_rmse, rf_mae, rf_r2]
})